# **Submission 1 Machine Learning Pipeline - Coronavirus Tweet Prediction**

### Import Library

In [2]:
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform, Trainer, Tuner, Evaluator, Pusher

from tfx.proto import example_gen_pb2, trainer_pb2, pusher_pb2
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing
import tensorflow_model_analysis as tfma
import os
import pandas as pd

### Set Variabel

In [3]:
PIPELINE_NAME = "mochyusuf-pipeline"
SCHEMA_PIPELINE_NAME = "coronavirus-schema"

PIPELINE_ROOT = PIPELINE_NAME
METADATA_PATH = os.path.join('metadata', PIPELINE_NAME, 'metadata.db')
SERVING_MODEL_DIR = os.path.join('serving_model', PIPELINE_NAME)

In [16]:
corona_train = pd.read_csv("data/Corona_NLP_train.csv", encoding='latin1')
corona_train.info()

corona_test = pd.read_csv("data/Corona_NLP_test.csv", encoding='latin1')
corona_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41157 entries, 0 to 41156
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   UserName       41157 non-null  int64 
 1   ScreenName     41157 non-null  int64 
 2   Location       32567 non-null  object
 3   TweetAt        41157 non-null  object
 4   OriginalTweet  41157 non-null  object
 5   Sentiment      41157 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.9+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3798 entries, 0 to 3797
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   UserName       3798 non-null   int64 
 1   ScreenName     3798 non-null   int64 
 2   Location       2964 non-null   object
 3   TweetAt        3798 non-null   object
 4   OriginalTweet  3798 non-null   object
 5   Sentiment      3798 non-null   object
dtypes: int64(2), object(4)
memory usage: 178.2+

In [17]:
data_path = "data"

corona_train = corona_train.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis = 1)
corona_train.to_csv(os.path.join(data_path, "Corona_NLP_train.csv"), index = False)

corona_test = corona_test.drop(["UserName", "ScreenName", "Location", "TweetAt"], axis = 1)
corona_test.to_csv(os.path.join(data_path, "Corona_NLP_test.csv"), index = False)


In [18]:
DATA_ROOT = 'data'
interactive_context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

### Data Ingestion

In [19]:
output = example_gen_pb2.Output(
    split_config = example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2)
    ])
)
example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output)

In [20]:
interactive_context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 4
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

### Data Validation

In [31]:
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
interactive_context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 8
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}))

In [ ]:
interactive_context.show(statistics_gen.outputs["statistics"])

In [32]:
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
interactive_context.run(schema_gen)

ExecutionResult(
    component_id: SchemaGen
    execution_id: 9
    outputs:
        schema: OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={}))

In [26]:
interactive_context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'OriginalTweet',BYTES,required,,-
'Sentiment',STRING,required,,'Sentiment'


,Values
Domain,
'Sentiment',"'Extremely Negative', 'Extremely Positive', 'Negative', 'Neutral', 'Positive'"


In [33]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"], schema=schema_gen.outputs["schema"]
)
interactive_context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 10
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}))

In [29]:
interactive_context.show(example_validator.outputs["anomalies"])

### Data Preprocessing

In [34]:
TRANSFORM_MODULE_FILE = "coronavirus_transform.py"
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs['schema'],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE)
)
interactive_context.run(transform)

ValueError: Specified module file 'd:\\Learn\\AI Mini Camp Dicoding\\Machine Learning Operations (MLOps)\\Submission 1\\Submission\\coronavirus_transform.py' for component Transform(spec: <tfx.types.standard_component_specs.TransformSpec object at 0x000001F44147F580>, executor_spec: <tfx.dsl.components.base.executor_spec.BeamExecutorSpec object at 0x000001F44147FFD0>, driver_class: <class 'tfx.dsl.components.base.base_driver.BaseDriver'>, component_id: Transform, inputs: {'examples': OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}), 'schema': OutputChannel(artifact_type=Schema, producer_component_id=SchemaGen, output_key=schema, additional_properties={}, additional_custom_properties={})}, outputs: {'transform_graph': OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}), 't